# Playing Deterministic Games

If you run the same game twice, the outcome will usually differ, because the deck is shuffled at random. To make a game reproducible, pass a `seed` to the game. The game has its own random number generator, which shuffles the deck and draws the first button, and `seed` initialises it. Games with the same seed, the same players and the same decisions deal exactly the same cards.

Players don't have access to that generator, so players that use random numbers can't change the cards that are dealt, but their own decisions aren't covered by `seed` either. If your players use the `random` module, seed it as well.

Running the following block of code again and again would always yield exactly the same outcome.

```{note}
Needless to say, if you are using LLMs as players, setting the seed only wouldn't guarantee determinism.
```

In [1]:
import random
from maverick import Game
from maverick.players import FoldBot, CallBot, AggressiveBot
from maverick import PlayerLike, PlayerState

# Seed the random module for players that use random numbers
random.seed(42)

# Seed the game's own random number generator, which deals the cards
game = Game(small_blind=10, big_blind=20, max_hands=2, seed=42)
players: list[PlayerLike] = [
    CallBot(name="CallBot"),
    AggressiveBot(name="AggroBot"),
    FoldBot(name="FoldBot"),
]

for player in players:
    game.add_player(player, state=PlayerState(stack=1000))
    
game.start()

# Print the players that are still in the game
print("\nNon-eliminated players:")
for snapshot in game.state.get_non_eliminated_players():
    print(f"  - {snapshot.name} is eligible for the next hand with {snapshot.state.stack} chips")


Non-eliminated players:
  - CallBot is eligible for the next hand with 1340 chips
  - AggroBot is eligible for the next hand with 680 chips
  - FoldBot is eligible for the next hand with 980 chips


Every game has a seed, even if you don't pass one: by default, it is drawn from the `random` module. You can read it from `game.seed` to replay a game you didn't seed explicitly. For the same reason, seeding `random` before creating the game, without passing `seed`, is reproducible too.

## Isolated players

With `player_isolation="process"` (see {doc}`player_isolation`), every player runs in its own worker process. Shuffling the deck and choosing the first button still happen in the engine, so `seed` still makes them reproducible. The workers don't inherit the random state of your script, so the game seeds the `random` module of every worker itself, from the game's seed, the player's UID and the hand number (see {ref}`isolated-player-random-numbers`). With the same `seed`, isolated players that use random numbers make the same decisions too.